In [4]:
# Set up the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "writethisdown"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# An empty query requests all documents
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB's ObjectID column if present
df.drop(columns=["_id"], inplace=True, errors="ignore")

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

image_filename = "logo.png"  # Replace with your own image
with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode()

app.layout = html.Div([
    html.Div(id="hidden-div", style={"display": "none"}),
    html.Div("Identifier -> Eric Sawyer", id="unique-identifier"),
    html.A(
    html.Img(src=f"data:image/png;base64,{encoded_image}", style={"width":"200px", "height":"200px"}),
    href="https://www.snhu.edu"),
    html.Center(html.B(html.H1("CS-340 Dashboard"))),
    html.Hr(),
    html.Div([
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Show All", "value": "all"},
            ],
            value="all",
            labelStyle={"display": "inline-block", "marginRight": "20px"},
        )
    ]),
    html.Hr(),
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": column, "id": column, "deletable": False, "selectable": True}
            for column in df.columns
        ],
        data=df.to_dict("records"),
        page_action="native",
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[],
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "100px",
            "maxWidth": "300px",
            "whiteSpace": "normal",
        },
        style_header={"fontWeight": "bold"},
    ),
    html.Br(),
    html.Hr(),
    # Show the chart and geolocation map side by side
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", className="col s12 m6"),
            html.Div(id="map-id", className="col s12 m6"),
        ],
    ),
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    filters = {
        "water": {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland",
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        },
        "mountain": {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler",
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        },
        "disaster": {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler",
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        },
    }

    query = filters.get(filter_type, {})
    filtered_records = db.read(query)
    filtered_df = pd.DataFrame.from_records(filtered_records)
    filtered_df.drop(columns=["_id"], inplace=True, errors="ignore")
    return filtered_df.to_dict("records")


# Display breeds represented in the data table
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")],
)
def update_graphs(view_data):
    if not view_data:
        return []

    chart_df = pd.DataFrame.from_records(view_data)
    if "breed" not in chart_df.columns:
        return []

    return [
        dcc.Graph(
            figure=px.pie(
                chart_df,
                names="breed",
                title="Animal Breeds",
            )
        )
    ]


# Highlight selected columns in the data table
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")],
)
def update_styles(selected_columns):
    return [
        {
            "if": {"column_id": column},
            "background_color": "#D2F3FF",
        }
        for column in (selected_columns or [])
    ]


# Update the geolocation map for the selected data entry
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(viewData, index):
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    
    if dff.empty or len(dff.columns) < 15:
        return []

    row = index[0] if index else 0
    
    if not isinstance(row, (int, np.integer)) or row < 0 or row >= len(dff):
        return []

    latitude = pd.to_numeric(dff.iloc[row, 13], errors="coerce")
    longitude = pd.to_numeric(dff.iloc[row, 14], errors="coerce")
    
    if pd.isna(latitude) or pd.isna(longitude):
        return []

    breed = dff.iloc[row, 4]
    name = dff.iloc[row, 9]

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([html.H1("Animal Name"), html.P(str(name))]),
                    ],
                ),
            ],
        )
    ]
  

# Run the app in JupyterLab
app.run_server()

Dash app running on https://currentdecimal-levelkoala-3000.codio.io/proxy/8050/
